In [ ]:
# Install required packages.
import os
import torch
os.environ['TORCH'] = torch.__version__
print(torch.__version__)

!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q git+https://github.com/pyg-team/pytorch_geometric.git
!pip install -q git+https://github.com/TorchSpatiotemporal/tsl.git


In [ ]:
import tsl
import torch
import numpy as np
import pandas as pd

print(f"tsl version  : {tsl.__version__}")
print(f"torch version: {torch.__version__}")

pd.options.display.float_format = '{:.2f}'.format
np.set_printoptions(edgeitems=3, precision=3)
torch.set_printoptions(edgeitems=2, precision=3)

# Utility functions ################
def print_matrix(matrix):
    return pd.DataFrame(matrix)

def print_model_size(model):
    tot = sum([p.numel() for p in model.parameters() if p.requires_grad])
    out = f"Number of model ({model.__class__.__name__}) parameters:{tot:10d}"
    print("=" * len(out))
    print(out)

In [ ]:
from tsl.datasets import MetrLA

dataset = MetrLA(root='./data')

print(dataset)

In [ ]:
print(f"Sampling period: {dataset.freq}")
print(f"Has missing values: {dataset.has_mask}")
print(f"Percentage of missing values: {(1 - dataset.mask.mean()) * 100:.2f}%")
print(f"Has exogenous variables: {dataset.has_covariates}")
print(f"Covariates: {', '.join(dataset.covariates.keys())}")

In [ ]:
print_matrix(dataset.dist)

In [ ]:
dataset.dataframe()

In [ ]:
print(f"Default similarity: {dataset.similarity_score}")
print(f"Available similarity options: {dataset.similarity_options}")
print("==========================================")

sim = dataset.get_similarity("distance")  # or dataset.compute_similarity()

print("Similarity matrix W:")
print_matrix(sim)

In [ ]:
connectivity = dataset.get_connectivity(threshold=0.1,
                                        include_self=False,
                                        normalize_axis=1,
                                        layout="edge_index")

In [ ]:
edge_index, edge_weight = connectivity

print(f'edge_index {edge_index.shape}:\n', edge_index)
print(f'edge_weight {edge_weight.shape}:\n', edge_weight)

In [ ]:
from tsl.ops.connectivity import edge_index_to_adj

adj = edge_index_to_adj(edge_index, edge_weight)
print(f'A {adj.shape}:')
print_matrix(adj)

In [ ]:
print(f'Sparse edge weights:\n', adj[edge_index[1], edge_index[0]])

In [ ]:
from tsl.data import SpatioTemporalDataset

torch_dataset = SpatioTemporalDataset(target=dataset.dataframe(),
                                      connectivity=connectivity,
                                      mask=dataset.mask,
                                      horizon=12,
                                      window=12,
                                      stride=1)
print(torch_dataset)

In [ ]:
sample = torch_dataset[0]
print(sample)

In [ ]:
sample.input.to_dict()

In [ ]:
sample.target.to_dict()

In [ ]:
if sample.has_mask:
    print(sample.mask)
else:
    print("Sample has no mask.")

In [ ]:
if sample.has_transform:
    print(sample.transform)
else:
    print("Sample has no transformation functions.")

In [ ]:
print(sample.pattern)
print("==================   Or we can print patterns and shapes together   ==================")
print(sample)

In [ ]:
batch = torch_dataset[:5]
print(batch)

In [ ]:
from tsl.data.datamodule import (SpatioTemporalDataModule,
                                 TemporalSplitter)
from tsl.data.preprocessing import StandardScaler

# Normalize data using mean and std computed over time and node dimensions
scalers = {'target': StandardScaler(axis=(0, 1))}

# Split data sequentially:
#   |------------ dataset -----------|
#   |--- train ---|- val -|-- test --|
splitter = TemporalSplitter(val_len=0.1, test_len=0.2)

dm = SpatioTemporalDataModule(
    dataset=torch_dataset,
    scalers=scalers,
    splitter=splitter,
    batch_size=64,
)

dm.setup()
print(dm)

In [ ]:
import torch.nn as nn

from tsl.nn.blocks.encoders import RNN
from tsl.nn.layers import NodeEmbedding, DiffConv
from einops.layers.torch import Rearrange  # reshape data with Einstein notation


class TimeThenSpaceModel(nn.Module):
    def __init__(self, input_size: int, n_nodes: int, horizon: int,
                 hidden_size: int = 32,
                 rnn_layers: int = 1,
                 gnn_kernel: int = 2):
        super(TimeThenSpaceModel, self).__init__()

        self.encoder = nn.Linear(input_size, hidden_size)

        self.node_embeddings = NodeEmbedding(n_nodes, hidden_size)

        self.time_nn = RNN(input_size=hidden_size,
                           hidden_size=hidden_size,
                           n_layers=rnn_layers,
                           cell='gru',
                           return_only_last_state=True)

        self.space_nn = DiffConv(in_channels=hidden_size,
                                 out_channels=hidden_size,
                                 k=gnn_kernel)

        self.decoder = nn.Linear(hidden_size, input_size * horizon)
        self.rearrange = Rearrange('b n (t f) -> b t n f', t=horizon)

    def forward(self, x, edge_index, edge_weight):
        # x: [batch time nodes features]
        x_enc = self.encoder(x)  # linear encoder: x_enc = xΘ + b
        x_emb = x_enc + self.node_embeddings()  # add node-identifier embeddings
        h = self.time_nn(x_emb)  # temporal processing: x=[b t n f] -> h=[b n f]
        z = self.space_nn(h, edge_index, edge_weight)  # spatial processing
        x_out = self.decoder(z)  # linear decoder: z=[b n f] -> x_out=[b n t⋅f]
        x_horizon = self.rearrange(x_out)
        return x_horizon

We can play with hyperparameters and make an instance of our model.

In [ ]:
hidden_size = 32   #@param
rnn_layers = 1     #@param
gnn_kernel = 2     #@param

input_size = torch_dataset.n_channels   # 1 channel
n_nodes = torch_dataset.n_nodes         # 207 nodes
horizon = torch_dataset.horizon         # 12 time steps

stgnn = TimeThenSpaceModel(input_size=input_size,
                           n_nodes=n_nodes,
                           horizon=horizon,
                           hidden_size=hidden_size,
                           rnn_layers=rnn_layers,
                           gnn_kernel=gnn_kernel)
print(stgnn)
print_model_size(stgnn)

In [ ]:
from tsl.metrics.torch import MaskedMAE, MaskedMAPE, MaskedMSE
from tsl.engines import Predictor

loss_fn = MaskedMAE()

metrics = {'mae': MaskedMAE(),
           'mape': MaskedMAPE(),
           'mse': MaskedMSE(),
           'mae_at_15': MaskedMAE(at=2),  # '2' indicates the third time step, # which correspond to 15 minutes ahead
           'mae_at_30': MaskedMAE(at=5),
           'mae_at_60': MaskedMAE(at=11),

           'mape_at_15': MaskedMAPE(at=2),
           'mape_at_30': MaskedMAPE(at=5),
           'mape_at_60': MaskedMAPE(at=11),

           'mse_at_15': MaskedMSE(at=2),
           'mse_at_30': MaskedMSE(at=5),
           'mse_at_60': MaskedMSE(at=11)

           }

# setup predictor
predictor = Predictor(
    model=stgnn,                   # our initialized model
    optim_class=torch.optim.Adam,  # specify optimizer to be used...
    optim_kwargs={'lr': 0.001},    # ...and parameters for its initialization
    loss_fn=loss_fn,               # which loss function to be used
    metrics=metrics                # metrics to be logged during train/val/test
)

In [ ]:
from pytorch_lightning.loggers import TensorBoardLogger

logger = TensorBoardLogger(save_dir="logs", name="tts", version=0)

In [ ]:
#%load_ext tensorboard
#%tensorboard --logdir logs

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint

checkpoint_callback = ModelCheckpoint(
    dirpath='logs',
    save_top_k=1,
    monitor='val_mae',
    mode='min',
)

trainer = pl.Trainer(max_epochs=50,
                     logger=logger,
                     accelerator= 'auto',
                    # limit_train_batches=100,  # end an epoch after 100 updates
                     callbacks=[checkpoint_callback])

trainer.fit(predictor, datamodule=dm)


In [ ]:
%reload_ext tensorboard
%tensorboard --logdir logs

In [ ]:
# Add any remaining globals (you might need to repeat this as errors show up)
from torchmetrics.metric import jit_distributed_available
torch.serialization.add_safe_globals([jit_distributed_available])

checkpoint = torch.load(checkpoint_callback.best_model_path, map_location='cpu', weights_only=False)
# Strip 'model.' prefix from keys
state_dict = {k.replace('model.', ''): v for k, v in checkpoint['state_dict'].items()}

# Load into the model
model = predictor.model  # already created by your Predictor
model.load_state_dict(state_dict)


predictor.freeze()
trainer.test(predictor, datamodule=dm)

In [ ]:
test_results = trainer.test(predictor, datamodule=dm)[0]
horizons_map = {
    "15 min ": {
        "mae": test_results.get('test_mae_at_15', float('nan')),
        "mape": test_results.get('test_mape_at_15', float('nan')),
        "mse": test_results.get('test_mse_at_15', float('nan'))
    },
    "30 min": {
        "mae": test_results.get('test_mae_at_30', float('nan')),
        "mape": test_results.get('test_mape_at_30', float('nan')),
        "mse": test_results.get('test_mse_at_30', float('nan'))
    },
    "60 min": {
        "mae": test_results.get('test_mae_at_60', float('nan')),
        "mape": test_results.get('test_mape_at_60', float('nan')),
        "mse": test_results.get('test_mse_at_60', float('nan'))
    }
}
performance_data = []
for horizon_label, mets in horizons_map.items():
    performance_data.append({
        "Prediction Horizon": horizon_label,
        "MAE": mets["mae"].item() if torch.is_tensor(mets["mae"]) else mets["mae"], # .item() if tensor
        "MAPE": mets["mape"].item() if torch.is_tensor(mets["mape"]) else mets["mape"],
        "MSE": mets["mse"].item() if torch.is_tensor(mets["mse"]) else mets["mse"],
    })

df_performance = pd.DataFrame(performance_data)
print("Average Performance for Model (across all sensors):")
print(df_performance)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 10))
sns.heatmap(adj, cmap='viridis')
plt.title("Adjacency Matrix Heatmap TTS")
plt.xlabel("Sensor Index")
plt.ylabel("Sensor Index")
plt.tight_layout()
plt.savefig("TTS_adjacency_matrix_heatmap.png")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df_melted = df_performance.melt(id_vars="Prediction Horizon", var_name="Metric", value_name="Value")

# Define metrics and set up subplots
metrics = ["MAE", "MAPE", "MSE"]
colors = ["#e39116" ,"#1b7fc6", "#25b325",]
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

# Plot each metric in a separate subplot
for i, metric in enumerate(metrics):
    ax = axes[i]
    subset = df_melted[df_melted["Metric"] == metric]
    sns.barplot(x="Prediction Horizon", y="Value", data=subset, ax=ax, color=colors[i])
    for p in ax.patches:
        height = p.get_height()
        ax.annotate(f'{height:.2f}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom', fontsize=10, color='black')

    ax.set_title(f"{metric}")
    ax.set_xlabel("Horizon")
    ax.set_ylabel(f"{metric} Value")
    ax.set_ylim(0, subset["Value"].max() * 1.2)
    ax.tick_params(axis='x', rotation=15)

# Final layout adjustments
plt.suptitle("AVERAGE TTS Model Performance Accross Sensors", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('Average_tts')
plt.show()

In [ ]:
model = predictor.model
model.eval()
device = next(model.parameters()).device

test_dataloader = dm.test_dataloader()
scaler = dm.scalers['target']

# Store full predictions and ground truths
all_y_true = []
all_y_pred = []

with torch.no_grad():
    for batch in test_dataloader:
        batch = batch.to(device)
        y_hat = model(x=batch.x, edge_index=batch.edge_index, edge_weight=batch.edge_weight)

        bs, H, N, F = y_hat.shape

        y_hat_rescaled = scaler.inverse_transform(y_hat.reshape(-1, F)).reshape(bs, H, N, F)
        y_true_rescaled = batch.y.reshape(-1, F).reshape(bs, H, N, F)

        # Store all predictions and ground truths
        all_y_pred.append(y_hat_rescaled.cpu())
        all_y_true.append(y_true_rescaled.cpu())

# Concatenate predictions across all batches
all_y_pred = torch.cat(all_y_pred, dim=0)  # [total_samples, H, N, F]
all_y_true = torch.cat(all_y_true, dim=0)

# We want to create one time series per sensor by flattening the samples and time horizon
T_total = all_y_pred.shape[0] * all_y_pred.shape[1]
all_y_pred_flat = all_y_pred.reshape(T_total, N, F)
all_y_true_flat = all_y_true.reshape(T_total, N, F)

# Plot actual vs predicted for the first 3 sensors
for sensor_idx in range(3):
    plt.figure(figsize=(16, 4))
    actual = all_y_true_flat[:, sensor_idx, 0].numpy()
    predicted = all_y_pred_flat[:, sensor_idx, 0].numpy()

    plt.plot(actual, label=f'Actual - Sensor {sensor_idx}', alpha=0.7)
    plt.plot(predicted, label=f'Predicted - Sensor {sensor_idx}', linestyle='--', alpha=0.7)

    plt.title(f'Sensor {sensor_idx} Actual vs Predicted Readings TTS')
    plt.xlabel('Time Steps (5 min each)')
    plt.ylabel('Traffic Flow (Speed)')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("tts_3sensors")
    plt.show()

In [ ]:
import torch
from tsl.nn.models.stgn import GraphWaveNetModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = GraphWaveNetModel(
    input_size=input_size,    # e.g., 1 channel
    output_size=input_size,   # usually same as input_size for forecasting
    n_nodes=n_nodes,          # e.g., 207 nodes
    horizon=horizon           # e.g., 12 time steps
).to(device)

print(model)


In [ ]:
from tsl.metrics.torch import MaskedMAE, MaskedMAPE, MaskedMSE
from tsl.engines import Predictor

loss_fn = MaskedMAE()

metrics = {'mae': MaskedMAE(),
           'mape': MaskedMAPE(),
           'mse': MaskedMSE(),
           'mae_at_15': MaskedMAE(at=2),  # '2' indicates the third time step, # which correspond to 15 minutes ahead
           'mae_at_30': MaskedMAE(at=5),
           'mae_at_60': MaskedMAE(at=11),

           'mape_at_15': MaskedMAPE(at=2),
           'mape_at_30': MaskedMAPE(at=5),
           'mape_at_60': MaskedMAPE(at=11),

           'mse_at_15': MaskedMSE(at=2),
           'mse_at_30': MaskedMSE(at=5),
           'mse_at_60': MaskedMSE(at=11)

           }

# setup predictor
predictor = Predictor(
    model= model,                   # our initialized model
    optim_class=torch.optim.Adam,  # specify optimizer to be used...
    optim_kwargs={'lr': 0.001},    # ...and parameters for its initialization
    loss_fn=loss_fn,               # which loss function to be used
    metrics=metrics                # metrics to be logged during train/val/test
)

In [ ]:
from pytorch_lightning.loggers import TensorBoardLogger

logger = TensorBoardLogger(save_dir="logs", name="gwn", version=0)

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

checkpoint_callback = ModelCheckpoint(
    dirpath='logs',
    save_top_k=1,
    monitor='val_mae',
    mode='min',
)
# Stop training early if val_mae doesn't improve for 10 epochs
early_stopping_callback = EarlyStopping(
    monitor='val_mae',
    patience=5,            # Number of epochs with no improvement
    mode='min',
    verbose=True
)
trainer = pl.Trainer(max_epochs=30,
                     logger=logger,
                     accelerator= 'auto',
                    # limit_train_batches=100,  # end an epoch after 100 updates
                     callbacks=[checkpoint_callback, early_stopping_callback])

trainer.fit(predictor, datamodule=dm)


In [ ]:
%reload_ext tensorboard
%tensorboard --logdir logs

In [ ]:
# Add any remaining globals (you might need to repeat this as errors show up)
from torchmetrics.metric import jit_distributed_available
torch.serialization.add_safe_globals([jit_distributed_available])

checkpoint = torch.load(checkpoint_callback.best_model_path, map_location='cpu', weights_only=False)
state_dict = {k.replace('model.', ''): v for k, v in checkpoint['state_dict'].items()}

# Load into the model
model = predictor.model  # already created by your Predictor
model.load_state_dict(state_dict)


predictor.freeze()
results = trainer.test(predictor, datamodule=dm)

In [ ]:
test_results = trainer.test(predictor, datamodule=dm)[0]
horizons_map = {
    "15 min ": {
        "mae": test_results.get('test_mae_at_15', float('nan')),
        "mape": test_results.get('test_mape_at_15', float('nan')),
        "mse": test_results.get('test_mse_at_15', float('nan'))
    },
    "30 min": {
        "mae": test_results.get('test_mae_at_30', float('nan')),
        "mape": test_results.get('test_mape_at_30', float('nan')),
        "mse": test_results.get('test_mse_at_30', float('nan'))
    },
    "60 min": {
        "mae": test_results.get('test_mae_at_60', float('nan')),
        "mape": test_results.get('test_mape_at_60', float('nan')),
        "mse": test_results.get('test_mse_at_60', float('nan'))
    }
}
performance_data = []
for horizon_label, mets in horizons_map.items():
    performance_data.append({
        "Prediction Horizon": horizon_label,
        "MAE": mets["mae"].item() if torch.is_tensor(mets["mae"]) else mets["mae"], # .item() if tensor
        "MAPE": mets["mape"].item() if torch.is_tensor(mets["mape"]) else mets["mape"],
        "MSE": mets["mse"].item() if torch.is_tensor(mets["mse"]) else mets["mse"],
    })

df_performance = pd.DataFrame(performance_data)
print("Average Performance for Model (across all sensors):")
print(df_performance)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Assuming adj is your adjacency matrix of shape (N, N)
adj_50 = adj[:50, :50]  # first 50 nodes

# Plot heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(adj_50, cmap='viridis')
plt.title("Adjacency Matrix Heatmap for First 50 Nodes (GWN)")
plt.xlabel("Sensor Index")
plt.ylabel("Sensor Index")
plt.tight_layout()
plt.savefig("GWN_adjacency_matrix_first_50_heatmap.png")
plt.show()

# Calculate influence scores
influence_scores = adj_50.sum(axis=1)
top_indices = np.argsort(influence_scores)[-15:][::-1]
top_scores = influence_scores[top_indices]

# Determine top 5 impacted nodes per influential node
results = []
for idx, score in zip(top_indices, top_scores):
    impacted_nodes = np.argsort(adj_50[idx])[-5:][::-1]  # Top 5 impacted
    impacted_weights = adj_50[idx][impacted_nodes]
    result = {
        "Influential_Node": idx,
        "Influence_Score": score,
    }
    for i in range(5):
        result[f"Impacted_Node_{i+1}"] = impacted_nodes[i]
        result[f"Weight_{i+1}"] = impacted_weights[i]
    results.append(result)

# Save to CSV
df = pd.DataFrame(results)
df.to_csv("top_15_influential_nodes.csv", index=False)

print("Saved results to 'top_15_influential_nodes.csv'")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df_melted = df_performance.melt(id_vars="Prediction Horizon", var_name="Metric", value_name="Value")

# Define metrics and set up subplots
metrics = ["MAE", "MAPE", "MSE"]
colors = ["#e39116" ,"#1b7fc6", "#25b325",]
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

# Plot each metric in a separate subplot
for i, metric in enumerate(metrics):
    ax = axes[i]
    subset = df_melted[df_melted["Metric"] == metric]
    sns.barplot(x="Prediction Horizon", y="Value", data=subset, ax=ax, color=colors[i])
    for p in ax.patches:
        height = p.get_height()
        ax.annotate(f'{height:.2f}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom', fontsize=10, color='black')

    ax.set_title(f"{metric}")
    ax.set_xlabel("Horizon")
    ax.set_ylabel(f"{metric} Value")
    ax.set_ylim(0, subset["Value"].max() * 1.2)
    ax.tick_params(axis='x', rotation=15)

# Final layout adjustments
plt.suptitle("AVERAGE GWN Model Performance Accross Sensors", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('Average_gwn')
plt.show()

In [ ]:
model = predictor.model
model.eval()
device = next(model.parameters()).device

test_dataloader = dm.test_dataloader()
scaler = dm.scalers['target']

# Store predictions and ground truths for only first 3 sensors
all_y_true = []
all_y_pred = []

with torch.no_grad():
    for batch in test_dataloader:
        batch = batch.to(device)
        y_hat = model(x=batch.x, edge_index=batch.edge_index, edge_weight=batch.edge_weight)

        bs, H, N, F = y_hat.shape

        # Keep only first 3 sensors (slice N dimension)
        y_hat_rescaled = scaler.inverse_transform(y_hat.reshape(-1, F)).reshape(bs, H, N, F)[:, :, :3, :]
        y_true_rescaled = batch.y.reshape(-1, F).reshape(bs, H, N, F)[:, :, :3, :]

        # Store
        all_y_pred.append(y_hat_rescaled.cpu())
        all_y_true.append(y_true_rescaled.cpu())

# Concatenate all batches
all_y_pred = torch.cat(all_y_pred, dim=0)  # [total_samples, H, 3, F]
all_y_true = torch.cat(all_y_true, dim=0)

# Flatten time horizon and samples for plotting
T_total = all_y_pred.shape[0] * all_y_pred.shape[1]
all_y_pred_flat = all_y_pred.reshape(T_total, 3, F)
all_y_true_flat = all_y_true.reshape(T_total, 3, F)

# Plot actual vs predicted for sensors 0, 1, 2
for sensor_idx in range(3):
    plt.figure(figsize=(16, 4))
    actual = all_y_true_flat[:, sensor_idx, 0].numpy()
    predicted = all_y_pred_flat[:, sensor_idx, 0].numpy()

    plt.plot(actual, label=f'Actual - Sensor {sensor_idx}', alpha=0.7)
    plt.plot(predicted, label=f'Predicted - Sensor {sensor_idx}', linestyle='--', alpha=0.7)

    plt.title(f'Sensor {sensor_idx} Actual vs Predicted Readings GWN')
    plt.xlabel('Time Steps (5 min each)')
    plt.ylabel('Traffic Flow (Speed)')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"gwn_sensor.png")
    plt.show()


In [ ]:
import torch
from tsl.nn.models.stgn import AGCRNModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'

input_size = torch_dataset.n_channels
n_nodes = torch_dataset.n_nodes
horizon = torch_dataset.horizon

model = AGCRNModel(
    input_size=input_size,
    output_size=input_size,
    n_nodes=n_nodes,
    horizon=horizon,
    emb_size=10,
    hidden_size=64,
    n_layers=1
).to(device)

print(model)

In [ ]:
from tsl.metrics.torch import MaskedMAE, MaskedMAPE, MaskedMSE
from tsl.engines import Predictor

loss_fn = MaskedMAE()

metrics = {'mae': MaskedMAE(),
           'mape': MaskedMAPE(),
           'mse': MaskedMSE(),
           'mae_at_15': MaskedMAE(at=2),  # '2' indicates the third time step, # which correspond to 15 minutes ahead
           'mae_at_30': MaskedMAE(at=5),
           'mae_at_60': MaskedMAE(at=11),

           'mape_at_15': MaskedMAPE(at=2),
           'mape_at_30': MaskedMAPE(at=5),
           'mape_at_60': MaskedMAPE(at=11),

           'mse_at_15': MaskedMSE(at=2),
           'mse_at_30': MaskedMSE(at=5),
           'mse_at_60': MaskedMSE(at=11)

           }

# setup predictor
predictor = Predictor(
    model= model,                   # our initialized model
    optim_class=torch.optim.Adam,  # specify optimizer to be used...
    optim_kwargs={'lr': 0.001},    # ...and parameters for its initialization
    loss_fn=loss_fn,               # which loss function to be used
    metrics=metrics                # metrics to be logged during train/val/test
)

In [ ]:
from pytorch_lightning.loggers import TensorBoardLogger

logger = TensorBoardLogger(save_dir="logs", name="agcrn", version=0)

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

checkpoint_callback = ModelCheckpoint(
    dirpath='logs',
    save_top_k=1,
    monitor='val_mae',
    mode='min',
)
# Stop training early if val_mae doesn't improve for 10 epochs
early_stopping_callback = EarlyStopping(
    monitor='val_mae',
    patience=10,            # Number of epochs with no improvement
    mode='min',
    verbose=True
)
trainer = pl.Trainer(max_epochs=50,
                     logger=logger,
                     accelerator= 'auto',
                    # limit_train_batches=100,  # end an epoch after 100 updates
                     callbacks=[checkpoint_callback, early_stopping_callback])

trainer.fit(predictor, datamodule=dm)


In [ ]:
%reload_ext tensorboard
%tensorboard --logdir logs

In [ ]:
from torchmetrics.metric import jit_distributed_available
torch.serialization.add_safe_globals([jit_distributed_available])

checkpoint = torch.load(checkpoint_callback.best_model_path, map_location='cpu', weights_only=False)
state_dict = {k.replace('model.', ''): v for k, v in checkpoint['state_dict'].items()}

# Load into the model
model = predictor.model  # already created by your Predictor
model.load_state_dict(state_dict)


predictor.freeze()
results = trainer.test(predictor, datamodule=dm)

In [ ]:
test_results = trainer.test(predictor, datamodule=dm)[0]
horizons_map = {
    "15 min ": {
        "mae": test_results.get('test_mae_at_15', float('nan')),
        "mape": test_results.get('test_mape_at_15', float('nan')),
        "mse": test_results.get('test_mse_at_15', float('nan'))
    },
    "30 min": {
        "mae": test_results.get('test_mae_at_30', float('nan')),
        "mape": test_results.get('test_mape_at_30', float('nan')),
        "mse": test_results.get('test_mse_at_30', float('nan'))
    },
    "60 min": {
        "mae": test_results.get('test_mae_at_60', float('nan')),
        "mape": test_results.get('test_mape_at_60', float('nan')),
        "mse": test_results.get('test_mse_at_60', float('nan'))
    }
}
performance_data = []
for horizon_label, mets in horizons_map.items():
    performance_data.append({
        "Prediction Horizon": horizon_label,
        "MAE": mets["mae"].item() if torch.is_tensor(mets["mae"]) else mets["mae"], # .item() if tensor
        "MAPE": mets["mape"].item() if torch.is_tensor(mets["mape"]) else mets["mape"],
        "MSE": mets["mse"].item() if torch.is_tensor(mets["mse"]) else mets["mse"],
    })

df_performance = pd.DataFrame(performance_data)
print("Average Performance for Model (across all sensors):")
print(df_performance)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df_melted = df_performance.melt(id_vars="Prediction Horizon", var_name="Metric", value_name="Value")

# Define metrics and set up subplots
metrics = ["MAE", "MAPE", "MSE"]
colors = ["#e39116" ,"#1b7fc6", "#25b325",]
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

# Plot each metric in a separate subplot
for i, metric in enumerate(metrics):
    ax = axes[i]
    subset = df_melted[df_melted["Metric"] == metric]
    sns.barplot(x="Prediction Horizon", y="Value", data=subset, ax=ax, color=colors[i])
    for p in ax.patches:
        height = p.get_height()
        ax.annotate(f'{height:.2f}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom', fontsize=10, color='black')

    ax.set_title(f"{metric}")
    ax.set_xlabel("Horizon")
    ax.set_ylabel(f"{metric} Value")
    ax.set_ylim(0, subset["Value"].max() * 1.2)
    ax.tick_params(axis='x', rotation=15)

# Final layout adjustments
plt.suptitle("AVERAGE AGCRN Model Performance Accross Sensors", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('Average_agcrn')
plt.show()

In [ ]:
model = predictor.model
model.eval()
device = next(model.parameters()).device

test_dataloader = dm.test_dataloader()
scaler = dm.scalers['target']

# Store predictions and ground truths for only first 3 sensors
all_y_true = []
all_y_pred = []

with torch.no_grad():
    for batch in test_dataloader:
        batch = batch.to(device)
        y_hat = model(x=batch.x)

        bs, H, N, F = y_hat.shape

        # Keep only first 3 sensors (slice N dimension)
        y_hat_rescaled = scaler.inverse_transform(y_hat.reshape(-1, F)).reshape(bs, H, N, F)[:, :, :3, :]
        y_true_rescaled = batch.y.reshape(-1, F).reshape(bs, H, N, F)[:, :, :3, :]

        # Store
        all_y_pred.append(y_hat_rescaled.cpu())
        all_y_true.append(y_true_rescaled.cpu())

# Concatenate all batches
all_y_pred = torch.cat(all_y_pred, dim=0)  # [total_samples, H, 3, F]
all_y_true = torch.cat(all_y_true, dim=0)

# Flatten time horizon and samples for plotting
T_total = all_y_pred.shape[0] * all_y_pred.shape[1]
all_y_pred_flat = all_y_pred.reshape(T_total, 3, F)
all_y_true_flat = all_y_true.reshape(T_total, 3, F)

# Plot actual vs predicted for sensors 0, 1, 2
for sensor_idx in range(3):
    plt.figure(figsize=(16, 4))
    actual = all_y_true_flat[:, sensor_idx, 0].numpy()
    predicted = all_y_pred_flat[:, sensor_idx, 0].numpy()

    plt.plot(actual, label=f'Actual - Sensor {sensor_idx}', alpha=0.7)
    plt.plot(predicted, label=f'Predicted - Sensor {sensor_idx}', linestyle='--', alpha=0.7)

    plt.title(f'Sensor {sensor_idx} Actual vs Predicted Readings AGCRN')
    plt.xlabel('Time Steps (5 min each)')
    plt.ylabel('Traffic Flow (Speed)')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"agcrn_sensor{sensor_idx}.png")
    plt.show()
